# 05 ISIC — Model Soup + TTA

Este notebook **no cambia la interfaz del experto**.  
Mantiene:

- misma arquitectura `EfficientNet-B3`
- mismo número de logits
- mismo `label_to_idx`
- misma salida final del experto

Hace dos cosas:

1. evalúa el **mejor checkpoint** tal cual y con **TTA horizontal flip**
2. si encuentra **2 o más checkpoints compatibles**, construye un **model soup** y lo evalúa también

> Si solo tienes **un checkpoint**, igual te sirve para probar **TTA** ahora mismo.


In [9]:

from pathlib import Path
import json
import random
from copy import deepcopy

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, f1_score, accuracy_score


In [10]:

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("No pude encontrar la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
WORKING_DIR = DATA_DIR / "working"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "isic_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# AJUSTA SOLO ESTO SI HACE FALTA
# ============================================================

# 1) checkpoint principal que ya sabes que funciona mejor
BEST_CKPT = OUTPUT_DIR / "isic_baseline_efficientnet_b3.pt"

# 2) candidatos extra para model soup
#    puedes dejar esta lista vacía, o llenarla con otros checkpoints compatibles
MANUAL_CKPTS = [
    # OUTPUT_DIR / "checkpoints" / "epoch_12.pt",
    # OUTPUT_DIR / "checkpoints" / "epoch_14.pt",
    # OUTPUT_DIR / "checkpoints" / "epoch_16.pt",
]

# 3) split de evaluación
SUBSET_PATH = WORKING_DIR / "isic" / "subsets" / "isic_subset_large.csv"

# 4) batch de inferencia
BATCH_SIZE = 32
NUM_WORKERS = 4

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DEVICE:", DEVICE)
print("BEST_CKPT:", BEST_CKPT)
print("SUBSET_PATH:", SUBSET_PATH)


PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
OUTPUT_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/isic_baseline
DEVICE: cuda
BEST_CKPT: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/isic_baseline/isic_baseline_efficientnet_b3.pt
SUBSET_PATH: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/isic/subsets/isic_subset_large.csv


In [11]:

if not BEST_CKPT.exists():
    raise FileNotFoundError(
        f"No encontré el checkpoint principal: {BEST_CKPT}\n"
        "Pon aquí la ruta exacta de tu mejor checkpoint."
    )

meta = torch.load(BEST_CKPT, map_location="cpu")

required_keys = ["label_to_idx", "image_size", "num_classes", "model_state_dict"]
missing = [k for k in required_keys if k not in meta]
if missing:
    raise KeyError(f"Al checkpoint le faltan estas claves: {missing}")

label_to_idx = {str(k): int(v) for k, v in meta["label_to_idx"].items()}
idx_to_label = {v: k for k, v in label_to_idx.items()}
IMAGE_SIZE = int(meta["image_size"])
NUM_CLASSES = int(meta["num_classes"])

print("IMAGE_SIZE:", IMAGE_SIZE)
print("NUM_CLASSES:", NUM_CLASSES)
print("label_to_idx:", label_to_idx)


IMAGE_SIZE: 300
NUM_CLASSES: 8
label_to_idx: {'AK': 0, 'BCC': 1, 'BKL': 2, 'DF': 3, 'MEL': 4, 'NV': 5, 'SCC': 6, 'VASC': 7}


In [12]:

subset_df = pd.read_csv(SUBSET_PATH)

if "target_text" not in subset_df.columns:
    if "target_label" in subset_df.columns:
        subset_df["target_text"] = subset_df["target_label"].astype(str)
    else:
        raise ValueError("No encontré ni target_text ni target_label.")

subset_df["target_text"] = subset_df["target_text"].astype(str)

allowed_labels = set(label_to_idx.keys())
subset_df = subset_df[subset_df["target_text"].isin(allowed_labels)].copy()

test_df = subset_df[subset_df["split_final"] == "test"].copy().reset_index(drop=True)
test_df["target_idx"] = test_df["target_text"].map(label_to_idx).astype(int)

print("test_df:", test_df.shape)
print(test_df["target_text"].value_counts())


test_df: (1500, 25)
target_text
NV      776
MEL     263
BCC     195
BKL     169
AK       44
SCC      32
VASC     11
DF       10
Name: count, dtype: int64


In [13]:

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform_hflip = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ISICDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

        # detectar automáticamente la columna de ruta
        candidate_cols = ["file_path", "image_path", "input_path", "path"]
        self.path_col = None
        for col in candidate_cols:
            if col in self.df.columns:
                self.path_col = col
                break

        if self.path_col is None:
            raise ValueError(
                f"No encontré columna de ruta. Columnas disponibles: {self.df.columns.tolist()}"
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row[self.path_col])
        if not image_path.is_absolute():
            image_path = PROJECT_ROOT / image_path

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        target = int(row["target_idx"])
        return image, target

test_dataset = ISICDataset(test_df, transform=eval_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)

print("Columna de ruta usada:", test_dataset.path_col)
print("test batches:", len(test_loader))


Columna de ruta usada: file_path
test batches: 47


In [14]:

def build_model(num_classes: int):
    weights = models.EfficientNet_B3_Weights.DEFAULT
    model = models.efficientnet_b3(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def extract_state_dict(ckpt_obj):
    if "model_state_dict" in ckpt_obj:
        return ckpt_obj["model_state_dict"]
    if "state_dict" in ckpt_obj:
        return ckpt_obj["state_dict"]
    raise KeyError("No encontré model_state_dict ni state_dict en el checkpoint.")

def load_model_from_ckpt(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state_dict = extract_state_dict(ckpt)

    if "label_to_idx" in ckpt:
        ckpt_map = {str(k): int(v) for k, v in ckpt["label_to_idx"].items()}
        if ckpt_map != label_to_idx:
            raise ValueError(f"{ckpt_path.name}: label_to_idx no coincide con el checkpoint base.")

    if "num_classes" in ckpt and int(ckpt["num_classes"]) != NUM_CLASSES:
        raise ValueError(f"{ckpt_path.name}: num_classes no coincide.")

    model = build_model(NUM_CLASSES)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)

    if missing or unexpected:
        print(f"[Aviso] {ckpt_path.name}: missing={missing}, unexpected={unexpected}")

    model = model.to(DEVICE)
    model.eval()
    return model


In [15]:

@torch.no_grad()
def predict_logits(model, loader):
    all_logits = []
    all_targets = []

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        all_logits.append(logits.detach().cpu())
        all_targets.append(targets.clone())

    all_logits = torch.cat(all_logits, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    return all_logits, all_targets

@torch.no_grad()
def predict_logits_tta_hflip(model, dataframe, batch_size=32, num_workers=4):
    ds_a = ISICDataset(dataframe, transform=eval_transform)
    ds_b = ISICDataset(dataframe, transform=eval_transform_hflip)

    loader_a = DataLoader(
        ds_a,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0)
    )
    loader_b = DataLoader(
        ds_b,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0)
    )

    logits_a, targets_a = predict_logits(model, loader_a)
    logits_b, targets_b = predict_logits(model, loader_b)

    if not torch.equal(targets_a, targets_b):
        raise ValueError("Los targets de TTA no coinciden. Revisa el orden del dataloader.")

    logits_mean = (logits_a + logits_b) / 2.0
    return logits_mean, targets_a

def metrics_from_logits(logits, targets):
    preds = logits.argmax(dim=1).numpy()
    targets_np = targets.numpy()

    return {
        "acc": float(accuracy_score(targets_np, preds)),
        "f1_macro": float(f1_score(targets_np, preds, average="macro", zero_division=0)),
        "f1_micro": float(f1_score(targets_np, preds, average="micro", zero_division=0)),
        "preds": preds,
        "targets": targets_np,
    }

def print_metrics_block(name, metrics_dict):
    print(f"\n=== {name} ===")
    print(f"acc      = {metrics_dict['acc']:.4f}")
    print(f"f1_macro = {metrics_dict['f1_macro']:.4f}")
    print(f"f1_micro = {metrics_dict['f1_micro']:.4f}")

    labels_order = list(range(NUM_CLASSES))
    target_names = [idx_to_label[i] for i in labels_order]

    print(classification_report(
        metrics_dict["targets"],
        metrics_dict["preds"],
        labels=labels_order,
        target_names=target_names,
        zero_division=0
    ))


In [16]:

# Evaluación del checkpoint principal
best_model = load_model_from_ckpt(BEST_CKPT)

best_logits_plain, best_targets = predict_logits(best_model, test_loader)
best_plain_metrics = metrics_from_logits(best_logits_plain, best_targets)
print_metrics_block("BEST CKPT - PLAIN", best_plain_metrics)

best_logits_tta, best_targets_tta = predict_logits_tta_hflip(
    best_model,
    test_df,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)
best_tta_metrics = metrics_from_logits(best_logits_tta, best_targets_tta)
print_metrics_block("BEST CKPT - TTA HFLIP", best_tta_metrics)



=== BEST CKPT - PLAIN ===
acc      = 0.6113
f1_macro = 0.4028
f1_micro = 0.6113
              precision    recall  f1-score   support

          AK       0.21      0.41      0.28        44
         BCC       0.50      0.70      0.58       195
         BKL       0.39      0.41      0.40       169
          DF       0.17      0.10      0.12        10
         MEL       0.48      0.49      0.49       263
          NV       0.86      0.71      0.78       776
         SCC       0.20      0.16      0.18        32
        VASC       0.26      0.82      0.39        11

    accuracy                           0.61      1500
   macro avg       0.38      0.47      0.40      1500
weighted avg       0.66      0.61      0.63      1500


=== BEST CKPT - TTA HFLIP ===
acc      = 0.6160
f1_macro = 0.4033
f1_micro = 0.6160
              precision    recall  f1-score   support

          AK       0.19      0.36      0.25        44
         BCC       0.50      0.66      0.57       195
         BKL       0

In [17]:

def clean_ckpt_list(paths):
    result = []
    seen = set()
    for p in paths:
        p = Path(p)
        if p.exists():
            rp = str(p.resolve())
            if rp not in seen:
                seen.add(rp)
                result.append(p)
    return result

candidate_ckpts = clean_ckpt_list([BEST_CKPT] + MANUAL_CKPTS)

print("Checkpoints candidatos:")
for p in candidate_ckpts:
    print("-", p)

if len(candidate_ckpts) < 2:
    print("\nSolo hay un checkpoint disponible. Model soup no se puede construir todavía.")


Checkpoints candidatos:
- /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/isic_baseline/isic_baseline_efficientnet_b3.pt

Solo hay un checkpoint disponible. Model soup no se puede construir todavía.


In [18]:

def build_uniform_model_soup(checkpoint_paths):
    checkpoint_paths = [Path(p) for p in checkpoint_paths]
    if len(checkpoint_paths) < 2:
        raise ValueError("Se necesitan al menos 2 checkpoints para model soup.")

    soups = []
    for ckpt_path in checkpoint_paths:
        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "label_to_idx" in ckpt:
            ckpt_map = {str(k): int(v) for k, v in ckpt["label_to_idx"].items()}
            if ckpt_map != label_to_idx:
                raise ValueError(f"{ckpt_path.name}: label_to_idx no coincide.")

        if "num_classes" in ckpt and int(ckpt["num_classes"]) != NUM_CLASSES:
            raise ValueError(f"{ckpt_path.name}: num_classes no coincide.")

        soups.append(extract_state_dict(ckpt))

    ref_keys = set(soups[0].keys())
    for i, sd in enumerate(soups[1:], start=1):
        if set(sd.keys()) != ref_keys:
            raise ValueError(f"El checkpoint {checkpoint_paths[i].name} no tiene las mismas claves.")

    soup_state = {}
    for key in soups[0].keys():
        tensors = [sd[key] for sd in soups]

        if not torch.is_tensor(tensors[0]):
            soup_state[key] = tensors[0]
            continue

        if tensors[0].dtype not in (
            torch.float16, torch.float32, torch.float64, torch.bfloat16
        ):
            soup_state[key] = tensors[0]
            continue

        stacked = torch.stack([t.float() for t in tensors], dim=0)
        soup_state[key] = stacked.mean(dim=0).to(tensors[0].dtype)

    return soup_state

if len(candidate_ckpts) >= 2:
    soup_state = build_uniform_model_soup(candidate_ckpts)

    soup_model = build_model(NUM_CLASSES)
    soup_model.load_state_dict(soup_state, strict=True)
    soup_model = soup_model.to(DEVICE)
    soup_model.eval()

    soup_logits_plain, soup_targets = predict_logits(soup_model, test_loader)
    soup_plain_metrics = metrics_from_logits(soup_logits_plain, soup_targets)
    print_metrics_block("MODEL SOUP - PLAIN", soup_plain_metrics)

    soup_logits_tta, soup_targets_tta = predict_logits_tta_hflip(
        soup_model,
        test_df,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS
    )
    soup_tta_metrics = metrics_from_logits(soup_logits_tta, soup_targets_tta)
    print_metrics_block("MODEL SOUP - TTA HFLIP", soup_tta_metrics)


In [19]:

# Guardar soup checkpoint si se construyó
if "soup_state" in globals():
    soup_ckpt_path = OUTPUT_DIR / "isic_baseline_efficientnet_b3_model_soup.pt"
    soup_metrics_path = OUTPUT_DIR / "isic_model_soup_test_metrics.json"

    soup_payload = {
        "model_name": "efficientnet_b3",
        "num_classes": NUM_CLASSES,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "image_size": IMAGE_SIZE,
        "model_state_dict": soup_state,
        "source_checkpoints": [str(p) for p in candidate_ckpts],
        "plain_test_acc": soup_plain_metrics["acc"],
        "plain_test_f1_macro": soup_plain_metrics["f1_macro"],
        "plain_test_f1_micro": soup_plain_metrics["f1_micro"],
        "tta_test_acc": soup_tta_metrics["acc"],
        "tta_test_f1_macro": soup_tta_metrics["f1_macro"],
        "tta_test_f1_micro": soup_tta_metrics["f1_micro"],
    }

    torch.save(soup_payload, soup_ckpt_path)

    with open(soup_metrics_path, "w", encoding="utf-8") as f:
        json.dump({
            "best_plain": {
                "acc": best_plain_metrics["acc"],
                "f1_macro": best_plain_metrics["f1_macro"],
                "f1_micro": best_plain_metrics["f1_micro"],
            },
            "best_tta_hflip": {
                "acc": best_tta_metrics["acc"],
                "f1_macro": best_tta_metrics["f1_macro"],
                "f1_micro": best_tta_metrics["f1_micro"],
            },
            "soup_plain": {
                "acc": soup_plain_metrics["acc"],
                "f1_macro": soup_plain_metrics["f1_macro"],
                "f1_micro": soup_plain_metrics["f1_micro"],
            },
            "soup_tta_hflip": {
                "acc": soup_tta_metrics["acc"],
                "f1_macro": soup_tta_metrics["f1_macro"],
                "f1_micro": soup_tta_metrics["f1_micro"],
            },
            "source_checkpoints": [str(p) for p in candidate_ckpts]
        }, f, ensure_ascii=False, indent=2)

    print("Soup checkpoint guardado en:", soup_ckpt_path)
    print("Métricas guardadas en:", soup_metrics_path)
else:
    print("No se guardó soup porque no había suficientes checkpoints compatibles.")


No se guardó soup porque no había suficientes checkpoints compatibles.


## Qué hacer según el resultado

### Caso A: solo tienes 1 checkpoint
Quédate con la comparación:
- `BEST CKPT - PLAIN`
- `BEST CKPT - TTA HFLIP`

Si `TTA HFLIP` mejora, ya puedes usar esa inferencia en evaluación del experto.

### Caso B: tienes 2 o más checkpoints compatibles
Compara:
- `BEST CKPT - PLAIN`
- `BEST CKPT - TTA HFLIP`
- `MODEL SOUP - PLAIN`
- `MODEL SOUP - TTA HFLIP`

Quédate con la mejor combinación **sin cambiar la interfaz del experto**.
